# Exp5.5.4 — Causal WHEN Phase Recovery

Analysis-only notebook. It reads finalized artifacts from `notebooks/artifacts/experiment_5_5_4_causal_when_phase_recovery/causal_when_phase_recovery_v1/`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_5_5_4_causal_when_phase_recovery' / 'causal_when_phase_recovery_v1'
runs = pd.read_csv(ART / 'runs.csv')
summary = pd.read_csv(ART / 'summary.csv')
paired = pd.read_csv(ART / 'paired_deltas.csv')
phase = pd.read_csv(ART / 'phase_metrics.csv')
confusion = pd.read_csv(ART / 'phase_confusion.csv')
routing = pd.read_csv(ART / 'routing_disagreement.csv')
margins = pd.read_csv(ART / 'margin_degradation.csv')
sensitivity = pd.read_csv(ART / 'phase_sensitivity.csv')
summary


## 1. Progress MAE and Spearman

In [ ]:
test_phase = phase[(phase['split'] == 'test') & (phase['when'] != 'oracle')]
agg = test_phase.groupby('when').agg(
    progress_mae=('sample_balanced_progress_mae', 'mean'),
    progress_spearman=('progress_spearman', 'mean'),
)
display(agg)
ax = agg['progress_mae'].plot(kind='bar', ylabel='Sample-balanced progress MAE', rot=0, title='GRU64 vs RSNN64 progress MAE')
plt.tight_layout()
plt.show()
ax = agg['progress_spearman'].plot(kind='bar', ylabel='Spearman', rot=0, title='GRU64 vs RSNN64 progress Spearman')
plt.tight_layout()
plt.show()


## 2. Phase10 and ±1-bin accuracy

In [ ]:
phase_acc = test_phase.groupby('when').agg(
    phase10=('sample_balanced_phase10_accuracy', 'mean'),
    within1=('sample_balanced_within1_bin_accuracy', 'mean'),
)
display(phase_acc)
phase_acc.plot(kind='bar', ylabel='Accuracy', rot=0, title='Relative10 phase recovery')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


## 3. Oracle / GRU / RSNN downstream test BA

In [ ]:
ba = runs.groupby(['when', 'routing'])['test_balanced_accuracy'].mean().unstack('routing')
display(ba)
ba.plot(kind='bar', ylabel='Test balanced accuracy', rot=0, title='Frozen Relative10 teacher replay')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
display(paired.groupby(['architecture', 'routing'])[['gap_oracle_minus_pred']].mean())


## 4. Routing error vs oracle-margin loss

In [ ]:
for (architecture, route), frame in margins.groupby(['architecture', 'routing']):
    fig, ax = plt.subplots(figsize=(6.0, 4.5))
    ax.scatter(frame['routing_disagreement_rate'], frame['margin_delta_pred_minus_oracle'], s=14, alpha=0.6)
    corr = frame[['routing_disagreement_rate', 'margin_delta_pred_minus_oracle']].corr().iloc[0, 1]
    ax.set_title(f'{architecture} / {route}: corr={corr:.3f}')
    ax.set_xlabel('Routing disagreement rate')
    ax.set_ylabel('Predicted margin - oracle margin')
    plt.tight_layout()
    plt.show()


## Additional diagnostics

In [ ]:
display(confusion.groupby(['architecture', 'true_phase', 'predicted_phase'])['count'].sum().reset_index().head(20))
display(routing.groupby('architecture')[['routing_disagreement_rate', 'mean_abs_bin_error', 'within1_bin_accuracy']].mean())
display(sensitivity.groupby(['true_phase', 'substitute_phase'])['mean_evidence_l2'].mean().unstack('substitute_phase'))
